# Kiểm tra dữ liệu sau khi chạy phase4: build model-ready-data

In [4]:
import duckdb
from pathlib import Path

con = duckdb.connect()
MODEL_DIR = Path("../data/processed/model_ready")

for name in [
    "train_model.parquet",
    "val_warm_model.parquet",
    "test_warm_model.parquet",
]:
    path = MODEL_DIR / name
    print(name)
    print(con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT user_idx) AS users,
            COUNT(DISTINCT item_idx) AS items,
            MIN(rating) AS min_rating,
            MAX(rating) AS max_rating
        FROM read_parquet('{str(path).replace("\\\\", "/")}')
    """).df())

train_model.parquet
      rows   users   items  min_rating  max_rating
0  5677563  752848  270719         1.0         5.0
val_warm_model.parquet
     rows   users   items  min_rating  max_rating
0  784495  307164  101199         1.0         5.0
test_warm_model.parquet
     rows   users  items  min_rating  max_rating
0  305558  157066  56121         1.0         5.0


kiểm tra overlap

In [5]:
print(con.execute(f"""
WITH train_pairs AS (
    SELECT DISTINCT user_idx, item_idx
    FROM read_parquet('{str(MODEL_DIR / "train_model.parquet").replace("\\\\", "/")}')
),
test_pairs AS (
    SELECT DISTINCT user_idx, item_idx
    FROM read_parquet('{str(MODEL_DIR / "test_warm_model.parquet").replace("\\\\", "/")}')
)
SELECT COUNT(*) AS overlap_pairs
FROM train_pairs t
INNER JOIN test_pairs x
ON t.user_idx = x.user_idx
AND t.item_idx = x.item_idx
""").df())

   overlap_pairs
0              0
